# Causally informed AE

The causally informed autoencoder adds a causal discovery-based penalty (using DirectLiNGAM) on the latent space during training so the autoencoder learns representations with causal structure.

*   The causal loss computed using DirectLiNGAM. The latent space Z is fed
into DirectLiNGAM. This produces a causal adjacency matrix A.
*   A penalty is added to the AE loss:
sparsity penalty → penalizes dense causal graphs
dagness penalty → penalizes cycles (non-DAG structure)
*   Training alternates between AE reconstruction and causal penalty

→ So the AE is nudged to produce a latent space that LiNGAM sees as a sparse causal DAG.

In [ ]:
# Ensure PyTorch is installed
try:
    import torch
    import torch.nn as nn
except ImportError:
    print("PyTorch not found. Installing PyTorch...")
    !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
    import torch
    import torch.nn as nn

# Redefine the Autoencoder model using PyTorch
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim, activation='relu'):
        super(Autoencoder, self).__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim

        # Define activation function based on string input
        if activation == 'relu':
            self.activation = nn.ReLU()
        elif activation == 'tanh':
            self.activation = nn.Tanh()
        elif activation == 'leakyrelu':
            self.activation = nn.LeakyReLU(negative_slope=0.01)
        elif activation == 'sigmoid':
            self.activation = nn.Sigmoid()
        elif activation == 'linear':
             self.activation = lambda x: x # Linear activation
        else:
            raise ValueError(f"Unknown activation function: {activation}")


        # Encoder layers (PyTorch)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            self.activation,
            nn.Linear(128, 64),
            self.activation,
            nn.Linear(64, latent_dim)
            #self.activation # Apply activation to latent space as well
        )

        # Decoder layers (PyTorch)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            self.activation,
            nn.Linear(64, 128),
            self.activation,
            nn.Linear(128, input_dim),
            nn.Tanh()  # Output activation (usually sigmoid or tanh for normalized data)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded # Return both reconstructed and latent representation

# The compute_causal_loss function is already compatible with PyTorch latent tensor
def compute_causal_loss(Z_latent, model_type="direct_lingam"):
    # Z_latent is a PyTorch tensor here
    Z_np = Z_latent.detach().cpu().numpy()

    # Run DirectLiNGAM (expects numpy array)
    model = DirectLiNGAM()
    model.fit(Z_np)
    A = model.adjacency_matrix_

    # Sparsity penalty (more edges = higher loss)
    sparsity_penalty = np.sum(np.abs(A)) / A.size

    # DAG-ness penalty (penalize cycles)
    # Create graph only for significant edges
    G = nx.from_numpy_array((np.abs(A) > 1e-3).astype(int), create_using=nx.DiGraph)
    try:
        # Check for cycles
        nx.find_cycle(G, orientation="original")
        dagness_penalty = 1.0
    except nx.NetworkXNoCycle:
        # No cycle found
        dagness_penalty = 0.0
    except Exception as e:
         # Handle other potential errors in graph creation/cycle detection
         print(f"Error in DAG check: {e}")
         dagness_penalty = 1.0 # Penalize if graph ops fail


    total_causal_penalty = sparsity_penalty + dagness_penalty
    # Return a PyTorch tensor
    return torch.tensor(total_causal_penalty, dtype=torch.float32)


# The train_causal_ae function (already using PyTorch)
def train_causal_ae(model, X, epochs=100, lr=1e-3, lambda_causal=1.0, causal_every=5):
    # X is a pandas DataFrame or numpy array, convert to PyTorch tensor
    X_tensor = torch.tensor(X.values if isinstance(X, pd.DataFrame) else X, dtype=torch.float32)

    model.train() # Set model to training mode
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        optimizer.zero_grad()
        # Autoencoder forward pass returns reconstruction and latent
        recon, Z = model(X_tensor)
        loss_recon = criterion(recon, X_tensor)

        # Causal penalty every few epochs
        if epoch % causal_every == 0:
            # Pass the PyTorch latent tensor to causal loss
            loss_causal = compute_causal_loss(Z)
        else:
            loss_causal = torch.tensor(0.0, device=X_tensor.device) # Ensure tensor is on the same device

        # Total loss
        loss_total = loss_recon + lambda_causal * loss_causal
        loss_total.backward()
        optimizer.step()

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"[Epoch {epoch+1}] MSE: {loss_recon.item():.4f} | Causal: {loss_causal.item():.4f} | Total: {loss_total.item():.4f}")

    model.eval()
    return model

input_dim = norm_df.shape[1]
model = Autoencoder(input_dim=input_dim, latent_dim=100, activation='leakyrelu')

# Train the model using the PyTorch training function
# Pass norm_df (pandas DataFrame or numpy array)
model = train_causal_ae(model, norm_df, epochs=100, lambda_causal=5.0)

print("\nTraining of Causal Autoencoder completed.")

In [ ]:
input_dim1 = norm_df1.shape[1]
model1 = Autoencoder(input_dim=input_dim1, latent_dim=100, activation='leakyrelu')

model1 = train_causal_ae(model1, norm_df1, epochs=100, lambda_causal=5.0)

In [ ]:
input_dim2 = norm_df2.shape[1]
model2 = Autoencoder(input_dim=input_dim2, latent_dim=50, activation='leakyrelu')

model2 = train_causal_ae(model2, norm_df2, epochs=100, lambda_causal=5.0)

In [ ]:
input_dim3 = norm_df3.shape[1]
model3 = Autoencoder(input_dim=input_dim3, latent_dim=100, activation='leakyrelu')

model3 = train_causal_ae(model3, norm_df3, epochs=100, lambda_causal=5.0)

In [ ]:
def get_latent_representation(model, data):
    model.eval()
    X_tensor = torch.tensor(data.values if isinstance(data, pd.DataFrame) else data, dtype=torch.float32)
    with torch.no_grad():
        latent = model.encoder(X_tensor)
    return latent.numpy()

reduced_data = get_latent_representation(model, norm_df)
print("Reduced data shape:", reduced_data.shape)

In [ ]:
from sklearn.metrics import mean_squared_error
import torch

# Convert the pandas DataFrame norm_df to a PyTorch tensor
norm_df_tensor = torch.tensor(norm_df.values, dtype=torch.float32)


model.eval()
with torch.no_grad():
    reconstructed_data_tensor, _ = model(norm_df_tensor)
    reconstructed_data = reconstructed_data_tensor.numpy()

mse = mean_squared_error(norm_df.values, reconstructed_data)

print(f"Mean Squared Error: {mse}")

In [ ]:
from sklearn.metrics import r2_score
r2 = r2_score(norm_df, reconstructed_data)
print("R-squared:", r2)

In [ ]:
reduced_data1 = get_latent_representation(model1, norm_df1)
print("Reduced data shape:", reduced_data1.shape)

In [ ]:
from sklearn.metrics import mean_squared_error
import torch

# Convert the pandas DataFrame norm_df to a PyTorch tensor
norm_df_tensor1 = torch.tensor(norm_df1.values, dtype=torch.float32)


model1.eval()
with torch.no_grad():
    reconstructed_data_tensor1, _ = model1(norm_df_tensor1)
    reconstructed_data1 = reconstructed_data_tensor1.numpy()

mse1 = mean_squared_error(norm_df1.values, reconstructed_data1)

print(f"Mean Squared Error: {mse1}")

In [ ]:
from sklearn.metrics import r2_score
r2 = r2_score(norm_df1, reconstructed_data1)
print("R-squared:", r2)

In [ ]:
reduced_data2 = get_latent_representation(model2, norm_df2)
print("Reduced data shape:", reduced_data2.shape)

In [ ]:
from sklearn.metrics import mean_squared_error
import torch

# Convert the pandas DataFrame norm_df to a PyTorch tensor
norm_df_tensor2 = torch.tensor(norm_df2.values, dtype=torch.float32)

model2.eval()
with torch.no_grad():
    reconstructed_data_tensor2, _ = model2(norm_df_tensor2)
    reconstructed_data2 = reconstructed_data_tensor2.numpy()

mse2 = mean_squared_error(norm_df2.values, reconstructed_data2)

print(f"Mean Squared Error: {mse2}")

In [ ]:
from sklearn.metrics import r2_score
r2 = r2_score(norm_df2, reconstructed_data2)
print("R-squared:", r2)

In [ ]:
reduced_data3 = get_latent_representation(model3, norm_df3)
print("Reduced data shape:", reduced_data3.shape)

In [ ]:
from sklearn.metrics import mean_squared_error
import torch

# Convert the pandas DataFrame norm_df to a PyTorch tensor
norm_df_tensor3 = torch.tensor(norm_df3.values, dtype=torch.float32)

model3.eval()
with torch.no_grad():
    reconstructed_data_tensor3, _ = model3(norm_df_tensor3)
    reconstructed_data3 = reconstructed_data_tensor3.numpy()

mse3 = mean_squared_error(norm_df3.values, reconstructed_data3)

print(f"Mean Squared Error: {mse3}")

In [ ]:
from sklearn.metrics import r2_score
r2 = r2_score(norm_df3, reconstructed_data3)
print("R-squared:", r2)